# Explainable Loan Default Risk Scoring System

## Phase 1: Data Exploration

### Project Objective

The objective of this project is to build an explainable machine learning system that predicts whether a customer is likely to default on a loan.

Rather than focusing only on prediction accuracy, this project also emphasizes:

- Understanding the data
- Identifying important features
- Building interpretable models
- Generating risk scores
- Explaining predictions using Explainable AI (SHAP)

This notebook focuses on understanding the dataset before any preprocessing or model development.

## Business Problem

Banks receive thousands of loan applications every day.

Approving loans for customers who later default results in financial losses, while rejecting reliable customers reduces business opportunities.

The objective is to build a model that helps estimate the probability of loan default using applicant information.

The final system will classify applicants into different risk categories and provide explanations for its predictions.

## Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display all columns when viewing the dataframe
pd.set_option("display.max_columns", None)

SyntaxError: unexpected character after line continuation character (1936779555.py, line 1)

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv("../data/Loan_Default.csv")

## Step 3: Display First Five Rows

In [ ]:
df.head()

## Step 4: Understand Dataset Structure

In [ ]:
df.shape


### Observation

In [ ]:
df.columns

In [ ]:
df.info()

### Observation

- The dataset contains **255,347 loan records**.
- It has **18 columns**, including customer attributes and the target variable.
- The dataset is sufficiently large for training machine learning models.

### Observation

- The dataset contains **255,347 records** and **18 columns**.
- There are **no missing values** in any column.
- The dataset contains a mix of **numerical** and **categorical** features.
- The `LoanID` column is an identifier and is not expected to contribute to prediction.
- The `Default` column is the target variable for the classification task.

In [ ]:
# Check duplicate rows
df.duplicated().sum()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Target variable distribution
df["Default"].value_counts()

In [ ]:
# Target variable percentage
df["Default"].value_counts(normalize=True) * 100

## Step 6: Visual Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt

default_counts = df["Default"].value_counts()

plt.figure(figsize=(6,4))
plt.bar(["No Default", "Default"], default_counts.values)

plt.title("Distribution of Loan Default")
plt.xlabel("Loan Status")
plt.ylabel("Number of Applicants")

plt.show()

### Observation

- Most applicants did not default on their loans.
- The dataset is clearly imbalanced, with approximately 88% non-default cases and 12% default cases.
- Accuracy alone may not be an appropriate evaluation metric because a model could achieve high accuracy by always predicting the majority class.

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["Age"], bins=20)

plt.title("Age Distribution of Applicants")
plt.xlabel("Age")
plt.ylabel("Number of Applicants")

plt.show()

### Observation

- Applicants are distributed across a wide age range from 18 to 69 years.
- No unrealistic age values are present.
- The dataset represents both younger and older borrowers.

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["Income"], bins=30)

plt.title("Income Distribution")
plt.xlabel("Annual Income")
plt.ylabel("Number of Applicants")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["CreditScore"], bins=25)

plt.title("Credit Score Distribution")
plt.xlabel("Credit Score")
plt.ylabel("Number of Applicants")

plt.show()

## Step 7: Outlier Analysis

In [ ]:
plt.figure(figsize=(8,5))

plt.boxplot(df["Income"], orientation="horizontal")

plt.title("Income Distribution (Box Plot)")
plt.xlabel("Income")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.boxplot(df["LoanAmount"], orientation="horizontal")

plt.title("Loan Amount Distribution (Box Plot)")
plt.xlabel("Loan Amount")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.boxplot(df["CreditScore"], orientation="horizontal")

plt.title("Credit Score Distribution (Box Plot)")
plt.xlabel("Credit Score")

plt.show()

## Step 8: Feature vs Target Analysis

In [ ]:
df.groupby("Default")["CreditScore"].mean()

In [ ]:
df.groupby("Default")["Income"].mean()

In [ ]:
df.groupby("Default")["LoanAmount"].mean()

In [ ]:
df.groupby("Default")["DTIRatio"].mean()

## Step 9: Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)

corr

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap")

plt.show()

### Observation

- The correlation heatmap shows the relationship among numerical features.
- Most variables have weak to moderate correlations.
- No pair of numerical variables appears to have extremely high correlation, indicating a low risk of multicollinearity.

## Step 10: Categorical Feature Analysis

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(data=df, x="Education")

plt.title("Education Distribution")
plt.xticks(rotation=20)

plt.show()

### Observation

- The dataset contains applicants from different educational backgrounds.
- Education may influence income level and repayment capability.

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(data=df, x="EmploymentType")

plt.title("Employment Type Distribution")

plt.show()

### Observation

- Applicants belong to different employment categories.
- Employment stability can affect loan repayment behavior.

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(data=df, x="MaritalStatus")

plt.title("Marital Status Distribution")

plt.show()

### Observation

- Applicants have different marital statuses.
- Family responsibilities may indirectly influence financial commitments.

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(data=df, x="LoanPurpose")

plt.title("Loan Purpose Distribution")

plt.xticks(rotation=30)

plt.show()

### Observation

- Loans are requested for multiple purposes.
- Loan purpose could influence the probability of default.

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="HasMortgage")

plt.title("Applicants with Mortgage")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="HasCoSigner")

plt.title("Applicants with Co-Signer")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="HasDependents")

plt.title("Applicants with Dependents")

plt.show()

## Step 11: Business Analysis - Default Rate by Category

In [ ]:
education_default = (
    df.groupby("Education")["Default"]
    .mean()
    .sort_values(ascending=False)
)

education_default

In [ ]:
plt.figure(figsize=(8,5))

education_default.plot(kind="bar")

plt.title("Default Rate by Education")
plt.ylabel("Default Rate")

plt.show()

In [ ]:
employment_default = (
    df.groupby("EmploymentType")["Default"]
    .mean()
    .sort_values(ascending=False)
)

employment_default

In [ ]:
plt.figure(figsize=(7,5))

employment_default.plot(kind="bar")

plt.title("Default Rate by Employment Type")
plt.ylabel("Default Rate")

plt.show()

In [ ]:
marital_default = (
    df.groupby("MaritalStatus")["Default"]
    .mean()
    .sort_values(ascending=False)
)

marital_default

In [ ]:
plt.figure(figsize=(7,5))

marital_default.plot(kind="bar")

plt.title("Default Rate by Marital Status")
plt.ylabel("Default Rate")

plt.show()

In [ ]:
loanpurpose_default = (
    df.groupby("LoanPurpose")["Default"]
    .mean()
    .sort_values(ascending=False)
)

loanpurpose_default

In [ ]:
plt.figure(figsize=(10,5))

loanpurpose_default.plot(kind="bar")

plt.title("Default Rate by Loan Purpose")
plt.ylabel("Default Rate")

plt.xticks(rotation=30)

plt.show()

# Step 12: Exploratory Data Analysis Summary

## Dataset Overview
- Total Records: 255,347
- Total Features: 18
- Target Variable: Default

## Data Quality
- No missing values were found.
- No duplicate records were found.
- Data types are appropriate for analysis.

## Target Variable
- Approximately 88% of applicants did not default.
- Approximately 12% of applicants defaulted.
- The dataset is imbalanced, which should be considered during model training.

## Numerical Features
- Credit Score, Income, Loan Amount, and DTI Ratio are important financial indicators.
- Numerical features contain outliers, but they appear realistic for financial data.

## Categorical Features
- The dataset contains multiple categorical variables including Education, Employment Type, Marital Status, Loan Purpose, Mortgage Status, Dependents, and Co-signer information.
- Most categorical features are fairly balanced because the dataset is synthetically generated.

## Business Insights
- Customer financial characteristics are expected to play a significant role in predicting loan default.
- Credit-related features and debt burden are likely to be among the most influential predictors.
- Further preprocessing, feature engineering, and explainable machine learning techniques will be applied in the next stages.

## Conclusion
The dataset is clean, complete, and suitable for building an explainable loan default prediction system.